# `stop_reason` Reference Guide

Every response from the Claude API includes a `stop_reason` field at the top level of the `Message` object.
It tells you **why** Claude stopped generating — and your agentic loop must branch on it.

```json
{
  "id": "msg_abc",
  "role": "assistant",
  "content": [ ... ],
  "stop_reason": "end_turn",   // ← this field
  "stop_sequence": null
}
```

## All 7 values at a glance

| `stop_reason` | Trigger | Correct action |
|---|---|---|
| `end_turn` | Claude finished naturally | Extract content and return |
| `max_tokens` | Output hit the `max_tokens` budget | Increase limit or implement continuation |
| `stop_sequence` | A string in `stop_sequences` was emitted | Check `response.stop_sequence` for which one |
| `tool_use` | Claude wants to call client-side tools | Execute tools, return **all** `tool_result` blocks in one `user` message |
| `pause_turn` | Server-side tool loop hit its iteration limit | Re-append current assistant content and send again |
| `refusal` | Content policy blocked generation | Surface the refusal gracefully; do not retry |
| `model_context_window_exceeded` | Input tokens exceed the model's context window | Prune oldest messages or run compaction |

> **Key distinction:** `stop_reason: "tool_use"` appears on the **response object**.
> The string `"tool_use"` also appears as `block.type` inside individual content blocks.
> Same string, two different structural positions — both are real and valid.


In [ ]:
import Anthropic from 'npm:@anthropic-ai/sdk';
import { load } from 'https://deno.land/std@0.220.0/dotenv/mod.ts';

await load({ envPath: '../.env', export: true });

const client = new Anthropic({ apiKey: Deno.env.get('ANTHROPIC_API_KEY') ?? '' });
const MODEL = 'claude-sonnet-4-6';

console.log('Setup complete. Model:', MODEL);

## 1. `end_turn` — Natural Completion

Claude finished its response without hitting any limit or triggering any special condition.
This is the normal case for conversational turns.

**How to handle:** Extract the text content and return it. No further API call needed.

In [ ]:
const endTurnResponse = await client.messages.create({
  model: MODEL,
  max_tokens: 256,
  messages: [{ role: 'user', content: 'What is the capital of France?' }],
});

console.log('stop_reason:', endTurnResponse.stop_reason); // "end_turn"

// ✅ Correct handler for end_turn
if (endTurnResponse.stop_reason === 'end_turn') {
  const text = endTurnResponse.content
    .filter((b): b is Anthropic.TextBlock => b.type === 'text')
    .map(b => b.text)
    .join('');
  console.log('Response:', text);
}

## 2. `max_tokens` — Output Budget Exhausted

Claude hit the `max_tokens` limit **before** finishing its response. The output is truncated,
possibly mid-sentence.

**Do not confuse with `model_context_window_exceeded`:**
- `max_tokens`: your *output* budget (the `max_tokens` parameter) ran out
- `model_context_window_exceeded`: the *input* itself is too long for the model's context window

**How to handle:** Two options:
1. **Increase `max_tokens`** if you simply set it too low
2. **Continuation pattern** — append the partial response as an assistant turn, then send a
   new user message asking Claude to continue from where it left off

In [ ]:
// Deliberately set max_tokens low to trigger truncation
const truncatedResponse = await client.messages.create({
  model: MODEL,
  max_tokens: 20,
  messages: [{ role: 'user', content: 'Explain the history of the internet in detail.' }],
});

console.log('stop_reason:', truncatedResponse.stop_reason); // "max_tokens"

const partial = truncatedResponse.content
  .filter((b): b is Anthropic.TextBlock => b.type === 'text')
  .map(b => b.text)
  .join('');
console.log('Partial output (truncated):', partial);

In [ ]:
// Continuation pattern: append partial response, then ask Claude to continue
async function withContinuation(userQuery: string, maxTokensPerTurn: number): Promise<string> {
  const messages: Anthropic.MessageParam[] = [{ role: 'user', content: userQuery }];
  let fullText = '';
  let turn = 0;

  while (true) {
    turn++;
    const response = await client.messages.create({
      model: MODEL,
      max_tokens: maxTokensPerTurn,
      messages,
    });

    const chunk = response.content
      .filter((b): b is Anthropic.TextBlock => b.type === 'text')
      .map(b => b.text)
      .join('');
    fullText += chunk;
    console.log(`Turn ${turn}: stop_reason=${response.stop_reason}, chunk_length=${chunk.length}`);

    if (response.stop_reason === 'end_turn') break;

    if (response.stop_reason === 'max_tokens') {
      // Append partial assistant response, then ask Claude to continue
      messages.push({ role: 'assistant', content: response.content });
      messages.push({ role: 'user', content: 'Continue from exactly where you left off.' });
      continue;
    }

    // Unexpected stop_reason — stop the loop
    break;
  }

  return fullText;
}

const fullResponse = await withContinuation(
  'List the first 5 US presidents with their terms.',
  80,
);
console.log('\nFull assembled response:');
console.log(fullResponse);

## 3. `stop_sequence` — Custom Stop Trigger

You passed `stop_sequences: ["STOP", "END"]` in the request and Claude emitted one of them.
The stop sequence string itself is **not** included in the response content — generation stops
just before it.

**`response.stop_sequence` (different field!)** tells you which sequence was hit.

**How to handle:** Inspect `response.stop_sequence` to decide what the trigger means.
Common use case: structured generation where each section ends with a sentinel.

In [ ]:
const stopSeqResponse = await client.messages.create({
  model: MODEL,
  max_tokens: 256,
  stop_sequences: ['[END]', '[PAUSE]'],
  messages: [{
    role: 'user',
    content:
      'List 3 fruits, one per line. After the last fruit, write [END] on its own line.',
  }],
});

console.log('stop_reason   :', stopSeqResponse.stop_reason);    // "stop_sequence"
console.log('stop_sequence :', stopSeqResponse.stop_sequence);  // "[END]"

const content = stopSeqResponse.content
  .filter((b): b is Anthropic.TextBlock => b.type === 'text')
  .map(b => b.text)
  .join('');
console.log('Content (stop sequence NOT included):');
console.log(content);

// ✅ Handler pattern
if (stopSeqResponse.stop_reason === 'stop_sequence') {
  const which = stopSeqResponse.stop_sequence;
  if (which === '[END]') {
    console.log('→ Section complete, process output');
  } else if (which === '[PAUSE]') {
    console.log('→ Intermediate pause, wait for more input');
  }
}

## 4. `tool_use` — Client-Side Tool Call

Claude wants to call one or more tools that **you** must execute.
When `stop_reason` is `"tool_use"`, the `content` array contains one or more blocks
with `type: "tool_use"`.

```json
// The same string "tool_use" appears in two places:
{ "stop_reason": "tool_use" }          // on the message  → WHY Claude stopped
{ "type": "tool_use", "id": "..." }    // on a content block → WHAT to call
```

**How to handle:**
1. Execute every `tool_use` block (concurrently with `Promise.all()` if independent)
2. Return **all** `tool_result` blocks in a **single** `user` message
3. For a failed tool, include `is_error: true` — never omit a result for a `tool_use` block
4. If any block has `is_error: true`, the subsequent request returns a 400 error

In [ ]:
// Simple tool: look up today's temperature
function getTemperature(city: string): string {
  const data: Record<string, number> = { Tokyo: 28, London: 17, 'New York': 24 };
  const temp = data[city];
  if (temp === undefined) throw new Error(`No data for city: ${city}`);
  return `${temp}°C`;
}

const temperatureTool: Anthropic.Tool = {
  name: 'get_temperature',
  description: 'Returns the current temperature for a given city.',
  input_schema: {
    type: 'object',
    properties: {
      city: { type: 'string', description: 'City name, e.g. "Tokyo"' },
    },
    required: ['city'],
  },
};

async function toolUseLoop(userQuery: string): Promise<void> {
  const messages: Anthropic.MessageParam[] = [{ role: 'user', content: userQuery }];

  while (true) {
    const response = await client.messages.create({
      model: MODEL,
      max_tokens: 512,
      tools: [temperatureTool],
      tool_choice: { type: 'auto' },
      messages,
    });

    console.log('stop_reason:', response.stop_reason);

    if (response.stop_reason === 'end_turn') {
      const text = response.content
        .filter((b): b is Anthropic.TextBlock => b.type === 'text')
        .map(b => b.text)
        .join('');
      console.log('Final answer:', text);
      break;
    }

    if (response.stop_reason === 'tool_use') {
      // Append assistant turn with tool_use blocks
      messages.push({ role: 'assistant', content: response.content });

      const toolUseBlocks = response.content.filter(
        (b): b is Anthropic.ToolUseBlock => b.type === 'tool_use',
      );

      // ── Fire all tools concurrently, collect in original order ──────────────
      const pending = toolUseBlocks.map(async (tc): Promise<Anthropic.ToolResultBlockParam> => {
        try {
          const result = getTemperature((tc.input as { city: string }).city);
          console.log(`  Tool ${tc.name}(${JSON.stringify(tc.input)}) → ${result}`);
          return { type: 'tool_result', tool_use_id: tc.id, content: result };
        } catch (err) {
          const msg = err instanceof Error ? err.message : String(err);
          console.log(`  Tool ${tc.name} FAILED: ${msg}`);
          // is_error: true is REQUIRED for failed tools — never omit the result
          return { type: 'tool_result', tool_use_id: tc.id, content: msg, is_error: true };
        }
      });

      const results = await Promise.all(pending);

      // All tool_result blocks go in ONE user message
      messages.push({ role: 'user', content: results });
      continue;
    }

    // Unexpected stop_reason — exit
    console.log('Unhandled stop_reason:', response.stop_reason);
    break;
  }
}

await toolUseLoop('What is the current temperature in Tokyo and London?');

## 5. `pause_turn` — Server-Side Tool Loop Limit

`pause_turn` is returned when **server-side tools** (e.g. computer use, extended web browsing)
are running inside a turn and hit their internal iteration limit. The turn is **not complete**;
Claude needs to continue.

This is different from `tool_use`, which is for *client-side* tools:

| `stop_reason` | Who executes the tool? | Turn complete? |
|---|---|---|
| `tool_use` | **You** (client code) | No — return `tool_result` |
| `pause_turn` | **Anthropic servers** | No — re-send the conversation |
| `end_turn` | N/A | **Yes** |

**How to handle:** Append the current assistant content as an `assistant` message,
then send the conversation again without a new user message.
Claude will resume from where it stopped.

In [ ]:
// pause_turn requires server-side tools (e.g. computer use) to trigger naturally.
// This mock simulates the conversation structure so the handler pattern is visible.

type MockApiResponse = {
  stop_reason: 'pause_turn' | 'end_turn';
  content: Anthropic.ContentBlock[];
};

let callCount = 0;

async function mockServerToolApi(
  _messages: Anthropic.MessageParam[],
): Promise<MockApiResponse> {
  callCount++;
  if (callCount === 1) {
    // Simulates hitting the server-side iteration limit mid-turn
    return {
      stop_reason: 'pause_turn',
      content: [{ type: 'text', text: 'Browsing the web... partial results so far: (iteration limit hit)' }],
    };
  }
  // Simulates Claude finishing after resume
  return {
    stop_reason: 'end_turn',
    content: [{ type: 'text', text: 'Research complete. Here are the full findings: ...' }],
  };
}

async function pauseTurnLoop(userQuery: string): Promise<void> {
  const messages: Anthropic.MessageParam[] = [{ role: 'user', content: userQuery }];
  let roundTrip = 0;

  while (true) {
    roundTrip++;
    const response = await mockServerToolApi(messages);
    console.log(`Round trip ${roundTrip}: stop_reason=${response.stop_reason}`);

    if (response.stop_reason === 'end_turn') {
      const text = response.content
        .filter((b): b is Anthropic.TextBlock => b.type === 'text')
        .map(b => b.text)
        .join('');
      console.log('Final:', text);
      break;
    }

    if (response.stop_reason === 'pause_turn') {
      // ✅ Re-append the current content and send again — the turn is NOT done
      messages.push({ role: 'assistant', content: response.content });
      // Do NOT add a new user message — just re-send to let the turn continue
      console.log('  → pause_turn: re-sending to continue the turn');
      continue;
    }

    break;
  }
}

await pauseTurnLoop('Research the latest AI benchmarks using web browsing.');

## 6. `refusal` — Content Policy Block

Claude's content policy explicitly blocked generation. The `stop_reason` is `"refusal"`
(not `"end_turn"`) and the response includes a `refusal` field with a brief explanation.

Note: many routine refusals (e.g. "I can't help with that") appear as normal text responses
with `stop_reason: "end_turn"`. The `"refusal"` stop_reason is reserved for hard policy blocks.

**How to handle:** Surface the refusal gracefully. Do not retry with the same input —
the content itself triggered the block. No amount of rephrasing will change a hard policy refusal.

In [ ]:
// Helper: safely call the API and handle refusals
async function safeChat(userMessage: string): Promise<void> {
  const response = await client.messages.create({
    model: MODEL,
    max_tokens: 256,
    messages: [{ role: 'user', content: userMessage }],
  });

  console.log('stop_reason:', response.stop_reason);

  if (response.stop_reason === 'refusal') {
    // Access the refusal reason — the 'refusal' field exists alongside 'content'
    const refusalText = (response as unknown as { refusal: string }).refusal ?? 'Request blocked by content policy';
    console.log('BLOCKED — Refusal reason:', refusalText);
    console.log('Action: surface to user without retry');
    return;
  }

  // Normal path (end_turn, etc.)
  const text = response.content
    .filter((b): b is Anthropic.TextBlock => b.type === 'text')
    .map(b => b.text)
    .join('');
  console.log('Response:', text.slice(0, 150));
}

// Normal request — expect end_turn
await safeChat('What is the speed of light?');

## 7. `model_context_window_exceeded` — Input Too Large

The **input** (system prompt + all messages + tool schemas) exceeds the model's context window
before generation even begins. This is an input-side problem, not an output limit.

**`max_tokens` vs `model_context_window_exceeded` — the key distinction:**

```
max_tokens                     → output budget exhausted while generating
model_context_window_exceeded  → input too large to process at all
```

**How to handle (choose one):**
1. **Prune oldest messages** — drop the earliest human/assistant turns
2. **Summarize** — replace early turns with a summary message
3. **Compaction** — use Claude to compress the conversation into a smaller form

The strategy depends on how much information from early turns is still relevant.

In [ ]:
// Context pruning: drop oldest human/assistant pairs when the window is exceeded.
// This is a simple drop-oldest strategy. For information-dense conversations,
// consider a summarization pass instead.

async function chatWithContextPruning(
  messages: Anthropic.MessageParam[],
  systemPrompt?: string,
): Promise<Anthropic.Message> {
  const working = [...messages];

  while (working.length > 0) {
    const response = await client.messages.create({
      model: MODEL,
      max_tokens: 512,
      system: systemPrompt,
      messages: working,
    });

    if (response.stop_reason === 'model_context_window_exceeded') {
      if (working.length <= 1) {
        throw new Error('Cannot prune further — single message already exceeds context window');
      }
      // Drop the two oldest turns (one human + one assistant = one exchange)
      working.splice(0, 2);
      console.log(`Context overflow — dropped oldest exchange. ${working.length} messages remain.`);
      continue;
    }

    return response;
  }

  throw new Error('All messages pruned — conversation cannot proceed');
}

// Simulate a conversation history that is already at the edge of the window
const conversation: Anthropic.MessageParam[] = [
  { role: 'user',      content: 'Tell me about machine learning.' },
  { role: 'assistant', content: 'Machine learning is a subfield of AI...' },
  { role: 'user',      content: 'What about deep learning specifically?' },
  { role: 'assistant', content: 'Deep learning uses neural networks with many layers...' },
  { role: 'user',      content: 'How does backpropagation work?' },
];

const result = await chatWithContextPruning(conversation);
console.log('stop_reason:', result.stop_reason);

const answer = result.content
  .filter((b): b is Anthropic.TextBlock => b.type === 'text')
  .map(b => b.text)
  .join('');
console.log('Answer:', answer.slice(0, 200));

## Summary — Agentic Loop Decision Framework

```
response.stop_reason
        |
        +── 'end_turn'                     Extract content → return to caller
        |
        +── 'max_tokens'                   Option A: increase max_tokens parameter
        |                                  Option B: continuation pattern
        |                                    → append partial as assistant turn
        |                                    → add new user message asking to continue
        |
        +── 'stop_sequence'                Check response.stop_sequence for which one
        |                                  → dispatch on the specific sentinel value
        |
        +── 'tool_use'                     Execute every tool_use block
        |   (client-side tools)              → Promise.all() for concurrent execution
        |                                    → return ALL tool_result in ONE user message
        |                                    → is_error: true for failed tools (never omit)
        |
        +── 'pause_turn'                   Turn is NOT done — server-side tool limit
        |   (server-side tools)              → append current content as assistant turn
        |                                    → re-send WITHOUT a new user message
        |
        +── 'refusal'                      Content policy block
        |                                    → do NOT retry — surface gracefully
        |
        +── 'model_context_window_exceeded' Input too large (not output)
                                             → prune oldest exchanges
                                             → or summarize / compact
```

### Key rules to internalize

| Rule | Detail |
|---|---|
| `tool_use` ≠ `type: 'tool_use'` | `stop_reason: 'tool_use'` is on the message; `type: 'tool_use'` is on individual content blocks — same string, different positions |
| Every `tool_use` needs a `tool_result` | Omitting a result for any `tool_use` block causes a 400 error on the next request |
| `is_error: true` is required for failures | Don't swallow exceptions — return them as `tool_result` with `is_error: true` |
| All `tool_result` blocks in ONE message | Splitting into separate user messages breaks parallel execution and "teaches" Claude to avoid parallel calls |
| `pause_turn` ≠ `end_turn` | The turn is still in progress — re-send to let Claude continue |
| `max_tokens` ≠ `model_context_window_exceeded` | Output budget vs input size — different problems, different fixes |

### Related notebooks

- `tool_use/tool_execution_strategy.ipynb` — parallel vs sequential tool execution in depth
- `tool_use/tool_choice.ipynb` — `auto` / `any` / `tool` / `none` modes
- `context_manage/context_management_guide.ipynb` — prompt caching, compaction, token budgets
- `tool_use/automatic-context-compaction.ipynb` — compressing history in long agentic workflows
